# Degradation robustness: InternVL3.5-8B vs Gemma 4 12B W4A16

How does extraction accuracy and throughput hold up as document image quality falls?

**Receipts only, by construction.** `degraded_20260811` is 55 receipts x 3 severity tiers.
Bank statements and invoices are never degraded -- SDG's rationale is that receipts are the
only document type users photograph; statements and invoices arrive as clean PDFs.

That suits the question: receipts are the one type where both models tie at 1.000 median F1
on clean images, so degradation is precisely the test that breaks the tie.

**Ground truth is identical across tiers** -- distortion never changes the answer. The only
variable is image quality.

| Tier | Suffix | Ink bleed | Paper | Warp | Camera |
|---|---|---|---|---|---|
| clean | -- | -- | -- | -- | -- |
| light | `_v1` | 0.05-0.15 | lighting gradient | 1-3%, +/-3 deg | blur 0.2-0.5, JPEG 85-95 |
| moderate | `_v2` | 0.15-0.30 | gradient + cast shadow | 3-6%, +/-8 deg | blur 0.4-0.8, JPEG 65-80 |
| heavy | `_v3` | 0.30-0.50 | fold + gradient + shadow | 6-10%, +/-14 deg | blur 0.7-1.3, JPEG 50-65 |

`heavy` is deliberately calibrated to be **hard but not impossible** -- the generator config
records tuning it back when a fold obscured the supplier name, on the principle that "a field
a careful human cannot read is measuring noise rather than robustness".

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 11,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        # Text as vector paths, so exported SVGs render identically on a
        # machine that lacks the font (PowerPoint, Keynote, Slides).
        "svg.fonttype": "path",
    }
)

# Categorical palette, validated rather than chosen by eye: CVD separation
# dE 20.0 (protan) / 29.4 (tritan), normal-vision dE 26.9 -- all well above the
# >=8 target. Colour follows the ENTITY: one fixed hue per model, never
# reassigned when a filter changes which series are drawn.
INTERNVL = "#4C72B0"
GEMMA = "#DD8452"
MODEL_COLOUR = {"InternVL3.5-8B": INTERNVL, "Gemma 4 12B W4A16": GEMMA}
MODELS = list(MODEL_COLOUR)

# The validator flags #DD8452 at 2.73:1 against a light surface -- below 3:1.
# That obligates relief, so every figure below carries direct value labels and
# every figure is backed by a table. It is not dismissable.

# Diverging ramp for the delta heatmap: the two model hues through a neutral
# grey midpoint, so a cell's colour still names the model it favours. Never a
# rainbow, and never a hue at the midpoint.
DELTA_CMAP = LinearSegmentedColormap.from_list("internvl_gemma", [INTERNVL, "#e8e8e6", GEMMA])

# Sequential (magnitude) ramp for the per-field heatmaps: one hue, light->dark.
FIELD_CMAP = "Blues"

TIERS = ["clean", "light", "moderate", "heavy"]
INK = "#2b2b2b"
MUTED = "#6b6b6b"

In [ ]:
import os

# Override with LMM_EVAL_ROOT to run this notebook against a copy of the results
# somewhere else (or against a fixture); defaults to the sandbox tree.
ROOT = Path(os.environ.get("LMM_EVAL_ROOT", "/home/jovyan/nfs_share/tod_2026/evaluation_data"))

# One run per model per condition. The clean runs cover all three document
# types, so receipts are filtered out of them below; the degraded runs are
# receipts only. Separate output dirs per run: read_completed_images() resumes
# from image_name, so a shared dir would make the second run skip everything.
RUNS = {
    ("InternVL3.5-8B", "clean"): ROOT / "output_internvl_synthetic",
    ("InternVL3.5-8B", "degraded"): ROOT / "output_internvl_degraded",
    ("Gemma 4 12B W4A16", "clean"): ROOT / "output_gemma_synthetic",
    ("Gemma 4 12B W4A16", "degraded"): ROOT / "output_gemma_degraded",
}

TIER_BY_SUFFIX = {"_v1": "light", "_v2": "moderate", "_v3": "heavy"}


def tier_of(image_name: str) -> str:
    """Recover the severity tier from the filename suffix."""
    stem = Path(image_name).stem
    for suffix, tier in TIER_BY_SUFFIX.items():
        if stem.endswith(suffix):
            return tier
    return "clean"


def locate(out_dir: Path, filename: str) -> Path:
    """Find a result file whether or not the evaluation/ level is present.

    The evaluate stage writes into <out_dir>/evaluation/, but a results folder
    copied off the box is often flattened. Accept either rather than making the
    reader reshape their download.
    """
    for candidate in (out_dir / "evaluation" / filename, out_dir / filename):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{filename} not found under {out_dir} (checked ./ and ./evaluation/)\n"
        "  Has that run finished? Each model needs a degraded run plus a clean run."
    )


def read_jsonl(path: Path) -> list[dict]:
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

In [ ]:
records: list[dict] = []

for (model, _condition), out_dir in RUNS.items():
    evaluations = read_jsonl(locate(out_dir, "evaluation_results.jsonl"))
    # processing_time is per image and lives with the extractions, not the scores.
    seconds = {
        r["image_name"]: r.get("processing_time")
        for r in read_jsonl(locate(out_dir, "cleaned_extractions.jsonl"))
    }

    for r in evaluations:
        if r.get("document_type") != "RECEIPT":
            continue  # clean runs carry all three types
        if r.get("error") or "median_f1" not in r:
            continue
        records.append(
            {
                "model": model,
                "tier": tier_of(r["image_name"]),
                "image_name": r["image_name"],
                "case": r["image_name"].split("_")[0],
                "overall_accuracy": r.get("overall_accuracy", np.nan),
                "median_f1": r.get("median_f1", np.nan),
                "seconds": seconds.get(r["image_name"], np.nan),
                "field_scores": r.get("field_scores", {}),
            }
        )

df = pd.DataFrame(records)
df["tier"] = pd.Categorical(df["tier"], categories=TIERS, ordered=True)

# Sanity: 55 receipts per model per tier. Anything else means a run is partial,
# or a degraded receipt was misclassified and dropped out of the RECEIPT filter.
counts = df.groupby(["model", "tier"], observed=True).size().unstack(fill_value=0)
counts

## Summary table

The table is not decoration -- the validated palette carries a contrast warning, so every
figure below is backed by readable numbers.

In [ ]:
summary = (
    df.groupby(["model", "tier"], observed=True)
    .agg(
        n=("image_name", "size"),
        overall_accuracy=("overall_accuracy", "mean"),
        median_f1=("median_f1", "mean"),
        sec_per_image=("seconds", "mean"),
    )
    .round(3)
)
summary

## Accuracy and throughput across tiers

**`overall_accuracy` is the primary metric.** `median_f1` is saturated for receipts -- both
models sit at exactly 1.000 on clean images and may stay pinned through the lighter tiers.
Leading with the saturated metric would draw two flat lines and hide the result.

Separate panels rather than a second y-axis: a dual-axis chart invites a comparison between
two incommensurable scales.

In [ ]:
fig_curves, axes = plt.subplots(1, 3, figsize=(16, 4.6))
fig_curves.suptitle(
    "Receipt extraction under image degradation  |  55 receipts x 4 tiers  |  2x L4, DP=2",
    fontsize=13,
    fontweight="bold",
    y=1.04,
)

for ax, metric, title, ylab in (
    (axes[0], "overall_accuracy", "Accuracy (primary)", "overall accuracy"),
    (axes[1], "median_f1", "Median F1 (saturated)", "median F1"),
):
    for rank, model in enumerate(MODELS):
        series = summary.loc[model, metric].reindex(TIERS)
        ax.plot(
            TIERS,
            series.values,
            marker="o",
            markersize=8,
            linewidth=2,
            color=MODEL_COLOUR[model],
            label=model,
            zorder=3,
        )
        # Direct label at the line end. Offset in opposite directions per
        # series so the two never overprint when the lines converge -- which
        # they do exactly when the result is most interesting.
        ax.annotate(
            f"{series.iloc[-1]:.3f}",
            xy=(len(TIERS) - 1, series.iloc[-1]),
            xytext=(7, 9 if rank == 0 else -9),
            textcoords="offset points",
            va="center",
            fontsize=10,
            fontweight="bold",
            color=INK,
        )
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylab)
    ax.set_ylim(0, 1.05)
    ax.legend(frameon=False, loc="lower left")

# Throughput is a magnitude comparison, so bars rather than a line.
ax = axes[2]
width = 0.38
x = np.arange(len(TIERS))
for offset, model in zip((-width / 2, width / 2), MODELS):
    vals = summary.loc[model, "sec_per_image"].reindex(TIERS)
    bars = ax.bar(
        x + offset,
        vals.values,
        width=width * 0.94,  # 2px-equivalent surface gap between adjacent bars
        color=MODEL_COLOUR[model],
        label=model,
        edgecolor="white",
        linewidth=1.2,
        zorder=3,
    )
    for bar, val in zip(bars, vals.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=INK,
        )
ax.set_xticks(x, TIERS)
ax.set_ylabel("seconds / image")
ax.set_title("Cost per image", fontsize=11)
# Headroom so the legend sits clear of the bars rather than over them.
ax.set_ylim(0, float(np.nanmax(summary["sec_per_image"].values)) * 1.45)
ax.legend(frameon=False, loc="upper center", ncols=2, fontsize=9)

for ax in axes:
    ax.tick_params(colors=MUTED)
    ax.set_xlabel("degradation tier")

plt.tight_layout()
plt.show()

## Which fields break first?

Degradation rarely hits every field equally. A hypothesis worth testing: `BUSINESS_ABN`
(11 digits, no redundancy -- one misread character fails the field) should degrade before
`SUPPLIER_NAME`, which a model can recover semantically from partial strokes.

Sequential ramp, one hue light-to-dark, because the quantity is magnitude. Every cell is
labelled, which is the convention for a matrix and also discharges the contrast warning.

In [ ]:
def field_matrix(model: str) -> pd.DataFrame:
    """Mean per-field F1 for one model, fields x tiers."""
    columns = {}
    for tier in TIERS:
        subset = df[(df["model"] == model) & (df["tier"] == tier)]
        collected: dict[str, list[float]] = {}
        for scores in subset["field_scores"]:
            for field, score in scores.items():
                collected.setdefault(field, []).append(score.get("f1_score", np.nan))
        columns[tier] = {f: float(np.nanmean(v)) for f, v in collected.items()}
    matrix = pd.DataFrame(columns).reindex(columns=TIERS)
    # Order fields by how far they fall, worst-hit at the top.
    return matrix.assign(_drop=matrix["clean"] - matrix["heavy"]).sort_values(
        "_drop", ascending=False
    ).drop(columns="_drop")


matrices = {model: field_matrix(model) for model in MODELS}

fig_fields, axes = plt.subplots(1, 2, figsize=(13, 7), sharey=True)
fig_fields.suptitle("Per-field F1 by degradation tier", fontsize=13, fontweight="bold", y=0.98)

for ax, model in zip(axes, MODELS):
    matrix = matrices[model].reindex(matrices[MODELS[0]].index)
    im = ax.imshow(matrix.values, cmap=FIELD_CMAP, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(TIERS)), TIERS)
    ax.set_yticks(range(len(matrix.index)), matrix.index, fontsize=9)
    ax.set_xlabel("degradation tier")
    # Title in ink; a coloured accent rule above the panel carries identity.
    # Text never wears the series colour.
    ax.set_title(model, fontsize=11, color=INK, fontweight="bold", pad=14)
    ax.add_patch(
        plt.Rectangle(
            (0, 1.012),
            1,
            0.014,
            transform=ax.transAxes,
            color=MODEL_COLOUR[model],
            clip_on=False,
            linewidth=0,
        )
    )
    ax.grid(False)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.values[i, j]
            if np.isnan(value):
                continue
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                # Ink on light cells, surface on dark ones -- text never wears
                # the series colour.
                color="white" if value > 0.6 else INK,
            )

fig_fields.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="mean F1")
plt.show()

## Where the models differ

Diverging ramp -- two hues through a neutral grey midpoint, with each pole in its model's
own colour, so a cell's hue still names the model it favours. Grey means the two are level.

In [ ]:
delta = (matrices["Gemma 4 12B W4A16"] - matrices["InternVL3.5-8B"]).reindex(
    matrices[MODELS[0]].index
)
limit = float(np.nanmax(np.abs(delta.values))) or 1.0

fig_delta, ax = plt.subplots(figsize=(7.5, 7))
im = ax.imshow(delta.values, cmap=DELTA_CMAP, vmin=-limit, vmax=limit, aspect="auto")
ax.set_xticks(range(len(TIERS)), TIERS)
ax.set_yticks(range(len(delta.index)), delta.index, fontsize=9)
ax.set_title(
    "Gemma minus InternVL, per field\n"
    "orange = Gemma ahead   ·   blue = InternVL ahead   ·   grey = level",
    fontsize=11,
    fontweight="bold",
)
ax.grid(False)
for i in range(delta.shape[0]):
    for j in range(delta.shape[1]):
        value = delta.values[i, j]
        if np.isnan(value):
            continue
        ax.text(j, i, f"{value:+.2f}", ha="center", va="center", fontsize=8, color=INK)

fig_delta.colorbar(im, ax=ax, fraction=0.04, pad=0.03, label="F1 difference")
ax.set_xlabel("degradation tier")
plt.tight_layout()
plt.show()

delta.round(3)

## Two scoring artefacts, and what they cost

The per-field view exposes two fields that fail for **both** models at **every** tier,
including clean. Neither is a perception failure:

**`BUSINESS_ADDRESS` -- a comma.** Ground truth is `384 Bailey Cres, Paddington QLD 4064`;
both models return `384 Bailey Cres Paddington QLD 4064`. The address is read perfectly and
scored 0.00, because the comparison is exact-match on a formatted string.

**`LINE_ITEM_PRICES` -- a semantic mismatch.** Ground truth distinguishes unit price
(`$27.13`) from line total (`$54.26 = 2 x $27.13`). Both models return the line total for
"prices". Gemma also duplicates it into `LINE_ITEM_TOTAL_PRICES` and so scores 1.00 there,
while InternVL leaves that field `NOT_FOUND` and scores 0.10 -- so Gemma's apparent lead on
that field is partly an artefact of which field it duplicated into, not better reading.

Both artefacts are **tier-independent and affect both models**, so they shift the level but
not the slope or the ranking. The table below shows the comparison is robust to them.

In [ ]:
EXCLUSIONS = {
    "all 14 fields": set(),
    "less BUSINESS_ADDRESS": {"BUSINESS_ADDRESS"},
    "less both artefacts": {"BUSINESS_ADDRESS", "LINE_ITEM_PRICES"},
}

rows = []
for model in MODELS:
    matrix = matrices[model]
    for tier in TIERS:
        row = {"model": model, "tier": tier}
        for label, excluded in EXCLUSIONS.items():
            keep = matrix.loc[~matrix.index.isin(excluded), tier]
            row[label] = round(float(np.nanmean(keep.values)), 3)
        rows.append(row)

adjusted = pd.DataFrame(rows).set_index(["model", "tier"])
retained = (
    adjusted.groupby("model", observed=True)
    .apply(lambda g: (g.loc[(slice(None), "heavy"), :].iloc[0] / g.loc[(slice(None), "clean"), :].iloc[0]))
    .round(3)
)
print("mean per-field F1")
display(adjusted)
print("\nfraction retained at heavy (heavy / clean)")
display(retained)

## Export for slides

SVG with text rendered as vector paths, so the figures survive a machine that lacks the
font. PNG alongside for anywhere SVG is awkward.

In [ ]:
FIG_DIR = Path(os.environ.get("LMM_FIG_DIR", "figures"))
FIG_DIR.mkdir(parents=True, exist_ok=True)

for name, figure in (
    ("degradation_curves", fig_curves),
    ("per_field_f1", fig_fields),
    ("per_field_delta", fig_delta),
):
    for extension in ("svg", "png"):
        path = FIG_DIR / f"{name}.{extension}"
        figure.savefig(path, format=extension, bbox_inches="tight", dpi=200)
        print(f"wrote {path}")

## Key takeaways

| Metric | InternVL3.5-8B | Gemma 4 12B W4A16 | Winner |
|---|---|---|---|
| Accuracy, clean | 0.761 | **0.813** | Gemma |
| Accuracy, light | 0.763 | **0.818** | Gemma |
| Accuracy, moderate | 0.746 | **0.806** | Gemma |
| Accuracy, heavy | 0.707 | **0.723** | Gemma (narrowly) |
| Retained at heavy | **92.9%** | 88.9% | InternVL |
| Seconds per image | 13.7 - 14.3 | **8.9 - 9.4** | Gemma, ~35% faster |
| Classification under degradation | 165/165 | 165/165 | tie |

**Neither model is meaningfully affected by light or moderate degradation.** Accuracy is flat
from clean through moderate -- Gemma 0.813 / 0.818 / 0.806, InternVL 0.761 / 0.763 / 0.746.
The apparent *rise* at `light` is a couple of field-scores across 770, i.e. noise. Only
`heavy` bites, and even there both retain ~90% of clean accuracy.

**Gemma is more accurate at every tier and about 35% faster**, and it holds that speed
advantage as quality falls -- degradation costs Gemma +5.2% time and InternVL +3.9%, so the
gap is stable rather than eroding.

**InternVL degrades more gracefully in relative terms** (92.9% vs 88.9% retained), but from a
lower base: it never overtakes. The two converge at `heavy` (0.723 vs 0.707), so the harder
the image, the less the model choice matters.

**Classification is untouched by degradation** -- 165/165 receipts correctly typed by both
models at every tier. Whatever degradation costs, it is not document identification.

## Confounds

State these wherever the numbers are quoted -- the comparison is controlled on data, ground
truth, scorer and hardware, but not on these:

- **Engine**: InternVL on vLLM 0.19.0, Gemma on 0.25.1.
- **Precision**: InternVL BF16, Gemma QAT W4A16.
- **Generation**: Gemma 4 is roughly a year newer (InternVL3.5 Aug 2025, Gemma 4 mid-2026),
  so part of any margin is generational rather than architectural.
- **Image handling**: InternVL uses 448-px pre-tiling (6-tile receipt budget); Gemma uses its
  own soft-token budget with pre-tiling off. Each is that model's correct configuration, not
  a handicap -- but they are not the same mechanism.

`enforce_eager` is matched (`false`, CUDA graphs on) for both, so throughput is comparable.

## Reading the result

Two outcomes would make the experiment uninformative rather than negative:

- **Both models hold at 1.000 through `heavy`** -- the tiers are too mild, and the finding is
  about the degradation config rather than the models.
- **Both collapse to near zero at `heavy`** -- the tier is too harsh to discriminate.

The tier was calibrated against rendered output to sit between those, but it was calibrated
for human legibility, not for these models.